# Read CSV from Unity Catalog Volume

Read a CSV file stored in a Unity Catalog volume and load it into a Spark DataFrame.

In [0]:
# Read a CSV file from a Unity Catalog volume into a DataFrame
# Replace the path with your actual volume path: /Volumes/<catalog>/<schema>/<volume>/<file_or_directory>

csv_path = "/Volumes/workspace/bronze/ingested_files/source_crm/sales_details.csv"

df = (
    spark.read
    .option("header", True)        # First row contains column names
    .option("inferSchema", True)    # Automatically infer column types
    .csv(csv_path)
)

display(df)

In [0]:
# Print the inferred schema and row count
print(df.schema)
print(f"Row count: {df.count()}")

### # **Store CSV data in Bronze database table (single table)** 

In [0]:
# Write the DataFrame into a new Unity Catalog table
# Uses Delta format (default in Databricks) with overwrite mode

table_name = "workspace.bronze.crm_sales_details"

(
    df.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_name)
)

print(f"Table '{table_name}' created successfully.")

# Verify by reading the table back
print(f"Row count: {spark.read.table(table_name).count()}")

**Read from Table or write a Sql to check data in table**

In [0]:
df = spark.read.table('workspace.bronze.crm_sales_details')
df.display()


In [0]:
#df= spark.option('create')read.csv('/Volumes/workspace/bronze/ingested_files/source_crm/cust_info_copy.csv')
df = (spark.read
        .option('header',True)
        .option('inferschema',True)
        .csv('/Volumes/workspace/bronze/ingested_files/source_crm/cust_info.csv')
     )
(
    df.write
    .mode('overwrite')
    .option('overwriteSchema',True)
    .saveAsTable('workspace.bronze.crm_cust_info')
)

In [0]:
df = (spark.read
        .option('header',True)
        .option('inferschema',True)
        .csv('/Volumes/workspace/bronze/ingested_files/source_erp/CUST_AZ12.csv')
     )
(
    df.write
    .mode('overwrite')
    .option('overwriteSchema',True)
    .saveAsTable('workspace.bronze.erp_cust_az12')
)

### Optimized approach to load multiple files:

In [0]:
INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/ingested_files/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/ingested_files/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/ingested_files/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/ingested_files/source_erp/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/ingested_files/source_erp/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/ingested_files/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]

In [0]:
for item in INGESTION_CONFIG:
    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")
    df = (spark.read
        .option('header',True)
        .option('inferschema',True)
        .csv(item["path"])
     )
    (
        df.write
        .mode('overwrite')
        .option('overwriteSchema',True)
        .saveAsTable(f"workspace.bronze.{item['table']}")
    )